# 01 – Exploratory Data Analysis

**Project:** Area Feasibility Scoring Model  
**Goal:** Understand the shape, quality and distribution of property price data before any feature engineering or modelling.

---
### Contents
1. Setup & Data Loading
2. Dataset Overview
3. Price Distribution Analysis
4. Area-Level Aggregation Preview
5. Affordability Baseline (raw)
6. Geospatial Overview (if coordinates available)
7. Key Takeaways

## 1 · Setup & Data Loading

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

from data_loader import generate_synthetic_data, aggregate_to_area

# -------------------------------------------------------------------
# Change this path to your real Land Registry CSV if available.
# We fall back to synthetic data automatically.
# -------------------------------------------------------------------
DATA_PATH   = '../data/pp-complete.csv'   # Land Registry Price Paid
USER_BUDGET = 300_000                     # £ – the user's property budget

if os.path.exists(DATA_PATH):
    from data_loader import load_land_registry
    tx_df = load_land_registry(DATA_PATH)
    print(f'Loaded {len(tx_df):,} real transactions')
else:
    print('Real data not found – using synthetic dataset.')
    tx_df = generate_synthetic_data(n_transactions=20_000, n_areas=50)

print(tx_df.shape)
tx_df.head()

## 2 · Dataset Overview

In [ ]:
tx_df.info()

In [ ]:
tx_df['price'].describe(percentiles=[.05, .25, .5, .75, .95])

In [ ]:
missing = tx_df.isnull().sum()
print('Missing values:')
print(missing[missing > 0] if missing.sum() else 'None – dataset is complete!')

## 3 · Price Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw price histogram
axes[0].hist(tx_df['price'].clip(upper=tx_df['price'].quantile(0.99)),
             bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(USER_BUDGET, color='red', linestyle='--', label=f'Budget £{USER_BUDGET:,.0f}')
axes[0].set_title('Property Price Distribution')
axes[0].set_xlabel('Price (£)')
axes[0].legend()

# Log-scale
axes[1].hist(np.log1p(tx_df['price']), bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Log Property Price Distribution')
axes[1].set_xlabel('log(1 + Price)')

plt.tight_layout()
plt.show()

pct_affordable = (tx_df['price'] <= USER_BUDGET).mean()
print(f'{pct_affordable:.1%} of all transactions are at or below the user budget')

## 4 · Area-Level Aggregation Preview

In [ ]:
area_df = aggregate_to_area(tx_df)
area_df.head(10)

In [ ]:
print(f'Number of distinct areas: {len(area_df)}')
area_df[['median_price', 'price_25th', 'price_75th', 'num_listings']].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
area_df_sorted = area_df.sort_values('median_price')

ax.fill_between(
    range(len(area_df_sorted)),
    area_df_sorted['price_25th'],
    area_df_sorted['price_75th'],
    alpha=0.3, color='steelblue', label='IQR (25th–75th)'
)
ax.plot(range(len(area_df_sorted)), area_df_sorted['median_price'],
        color='steelblue', label='Median price')
ax.axhline(USER_BUDGET, color='red', linestyle='--', label=f'User budget £{USER_BUDGET:,.0f}')
ax.set_xlabel('Area (sorted by median price)')
ax.set_ylabel('Price (£)')
ax.set_title('Area Median Price vs User Budget')
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Affordability Baseline (raw)

In [ ]:
area_df['affordable'] = (USER_BUDGET >= area_df['median_price']).astype(int)

counts = area_df['affordable'].value_counts().rename({0: 'Not Affordable', 1: 'Affordable'})
print('Area-level class distribution:')
print(counts)
print(f'\nBaseline affordability rate: {area_df["affordable"].mean():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class bar chart
counts.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Affordability Class Distribution')
axes[0].set_xticklabels(counts.index, rotation=0)
axes[0].set_ylabel('Number of Areas')

# Affordability ratio distribution
area_df['affordability_ratio'] = USER_BUDGET / area_df['median_price']
axes[1].hist(area_df['affordability_ratio'], bins=30, color='purple', edgecolor='white')
axes[1].axvline(1.0, color='red', linestyle='--', label='Ratio = 1 (breakeven)')
axes[1].set_title('Affordability Ratio Distribution\n(budget / median_price)')
axes[1].set_xlabel('Affordability Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6 · Geospatial Overview (optional)

In [ ]:
if 'latitude' in area_df.columns and 'longitude' in area_df.columns:
    fig, ax = plt.subplots(figsize=(8, 7))
    sc = ax.scatter(
        area_df['longitude'], area_df['latitude'],
        c=area_df['median_price'], cmap='YlOrRd', s=30, alpha=0.7
    )
    plt.colorbar(sc, ax=ax, label='Median Price (£)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Median Price by Location')
    plt.tight_layout()
    plt.show()
else:
    print('No geospatial columns found – skipping map.')

## 7 · Key Takeaways

_Update after running:_

- **Dataset:** N transactions across M areas
- **Price range:** £… – £… (median £…)
- **Affordability rate:** % of areas where budget ≥ median price
- **Class balance:** roughly balanced / imbalanced?
- **Outliers:** any extreme prices to consider?
- **Next step:** → `02_feature_engineering.ipynb`